## Format and load the RFC and CPN basin locations to the warehouse

In [ ]:
import geopandas as gpd

from teehr import RemoteReadWriteEvaluation
from teehr.evaluation.spark_session_utils import create_spark_session
from pathlib import Path

### Re-format and save the CPN basins provided by Bob Lounsbury (USBR)
The original layer is in Teams at: `"Rec-FVED-Internal - Documents\Project Work\GIS\Data\SnowPolygonsfromBob_2026-07-31.zip"`

In [ ]:
gdf = gpd.read_file("~/git/teehr-fved/warehouse/remote/02_loading/07-31-2026 Polygons from Bob Lounsbury/CPN_Forecast_DAs_02MAR2026.shp", encoding="cp1252")
gdf["id"] = gdf["Name"].str.split().str[0]
gdf.rename(columns={"Name": "name"}, inplace=True)
gdf["properties"] = [{"source": "Bob Lounsbury, USBR", "date": "2026-07-31"}] * len(gdf)
gdf["id"] = "cpnbasin-" + gdf["id"]
gdf = gdf[["id", "name", "geometry", "properties"]]
gdf.head()

In [ ]:
gdf.to_parquet("cpn_forecast_locations.parquet")

In [ ]:
spark = create_spark_session()
ev = RemoteReadWriteEvaluation(spark=spark, enable_spark_proxy=True)

#### Load to the warehouse

In [ ]:
%%time
ev.locations.load_spatial("cpn_forecast_locations.parquet")

### Format and load the RFC basins
Read the layer `"Rec-FVED-Internal - Documents\Project Work\GIS\Data\RFC_BasinBoundaries\conus_scale\rfc_basins_zones_usbr_location_id_attrs.parquet" (see readme in the Teams dir for notes on how it was created)

In [ ]:
gdf = gpd.read_parquet("rfc_basins_zones_usbr_location_id_attrs.parquet")
gdf.rename(columns={"location_id": "id"}, inplace=True)
gdf["name"] = gdf["RFC"] + "-" + gdf["id"]
gdf = gdf[["id", "name", "geometry", "properties"]]
gdf["id"] = "rfcbasin-" + gdf["id"]
gdf.head()

In [ ]:
gdf.to_parquet("rfc_basin_locations.parquet")

#### Load to the warehouse

In [ ]:
%%time
ev.locations.load_spatial("rfc_basin_locations.parquet")

In [ ]:
ev.locations.show()

In [ ]:
spark.stop()